# 06b - Pipeline Definition: Service Orders (Tabular)

**Azure ML MLOps Workshop | Track B - Tabular Data**

### Context

In Notebook 06, you defined a training pipeline for the text classifier.
Now you'll build the **same pipeline structure** for the tabular model —
demonstrating that pipelines are reusable patterns, not one-off configurations.

### What you'll do:
1. Define a pipeline component for tabular training
2. Compose the pipeline using the `@dsl.pipeline` decorator
3. Submit and compare pipeline runs

### Key difference from Track A:
- **Track A** pipeline: `preprocess.py` -> `train.py` (text)
- **Track B** pipeline: `preprocess_os.py` -> `train_os.py` (tabular)
- **Same pipeline YAML structure:** swap the data input and scripts, everything else stays the same

---

In [ ]:
from azure.ai.ml import MLClient, Input, Output, command, dsl
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Define Pipeline Components

In [ ]:
from azure.ai.ml.entities import Environment

custom_env = Environment(
    name="contoso-training-env",
    description="Training environment for Contoso classifiers (text and tabular)",
    conda_file="../../environment/conda.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
)

custom_env = ml_client.environments.create_or_update(custom_env)
print(f"Environment registered: {custom_env.name} v{custom_env.version}")

In [ ]:
train_component = command(
    name="train_repair_classifier",
    display_name="Train Contoso Repair Classifier (Tabular)",
    description="Trains a tabular classifier on service orders to predict repair type.",
    inputs={
        "input_data": Input(type="uri_file"),
        "model_name": Input(type="string", default="logistic_regression"),
        "max_categories": Input(type="integer", default=20),
        "test_size": Input(type="number", default=0.2),
    },
    outputs={
        "model_output": Output(type="uri_folder"),
    },
    code="../../src/track_b_tabular",
    command=(
        "python train_os.py "
        "--input-data ${{inputs.input_data}} "
        "--model-name ${{inputs.model_name}} "
        "--max-categories ${{inputs.max_categories}} "
        "--test-size ${{inputs.test_size}} "
        "--output-dir ${{outputs.model_output}}"
    ),
    environment=f"{custom_env.name}:{custom_env.version}",
)

print(f"Training component defined: {train_component.name}")

## Step 2: Define the Pipeline

Compare this with the Track A pipeline — the structure is nearly identical.
Only the component, data input, and parameter names change.

In [ ]:
@dsl.pipeline(
    name="contoso-repair-classifier-pipeline",
    description="End-to-end training pipeline for Contoso service orders repair classification.",
    compute="cpu-cluster",
)
def repair_classifier_pipeline(
    training_data: Input,
    model_type: str = "logistic_regression",
    max_categories: int = 20,
):
    train_step = train_component(
        input_data=training_data,
        model_name=model_type,
        max_categories=max_categories,
        test_size=0.2,
    )
    train_step.display_name = f"Train {model_type}"

    return {"model_output": train_step.outputs.model_output}

print("Pipeline function defined.")

## Step 3: Submit the Pipeline Job

In [ ]:
pipeline_job = repair_classifier_pipeline(
    training_data=Input(
        type="uri_file",
        path="azureml:service-orders:2",
    ),
    model_type="logistic_regression",
    max_categories=20,
)

pipeline_job.experiment_name = "contoso-repair-classifier-pipeline"

submitted_job = ml_client.jobs.create_or_update(pipeline_job)
print(f"Pipeline submitted!")
print(f"  Job name: {submitted_job.name}")
print(f"  Status: {submitted_job.status}")
print(f"  Studio URL: {submitted_job.studio_url}")

In [ ]:
# Wait for completion (optional)
ml_client.jobs.stream(submitted_job.name)

## Step 4: Submit Multiple Pipeline Runs

In [ ]:
model_types = ["random_forest", "gradient_boosting"]

for model_type in model_types:
    pipeline_job = repair_classifier_pipeline(
        training_data=Input(
            type="uri_file",
            path="azureml:service-orders:2",
        ),
        model_type=model_type,
        max_categories=20,
    )
    pipeline_job.experiment_name = "contoso-repair-classifier-pipeline"

    submitted = ml_client.jobs.create_or_update(pipeline_job)
    print(f"Submitted pipeline for {model_type}: {submitted.name}")

## Go to Azure ML Studio Now

Navigate to **Jobs** and you should see **two pipeline experiments**:

| Pipeline | Data | Scripts |
|----------|------|--------|
| `contoso-lead-classifier-pipeline` | `classified-inspections` | `preprocess.py`, `train.py` |
| `contoso-repair-classifier-pipeline` | `service-orders` | `preprocess_os.py`, `train_os.py` |

Click on each to compare the DAG visualizations side by side.

---

## The Full Dual-Track MLOps Picture

```
Track A (Text)                          Track B (Tabular)
──────────────                          ─────────────────
classified-inspections v1,v2         service-orders v1,v2
        │                                       │
        v                                       v
contoso-lead-classifier experiment        contoso-repair-classifier experiment
        │                                       │
        v                                       v
contoso-lead-classifier model             contoso-repair-classifier model
        │                                       │
        v                                       v
contoso-lead-classifier endpoint          contoso-repair-classifier endpoint
        │                                       │
        v                                       v
contoso-lead-monitor                      contoso-repair-monitor
        │                                       │
        v                                       v
contoso-lead-classifier-pipeline          contoso-repair-classifier-pipeline
```

Two completely different ML problems. Same MLOps platform. Same patterns. Same APIs.

---

## Key Takeaways

| Concept | What We Did |
|---------|------------|
| **Reusable patterns** | Swap data + scripts, keep the pipeline structure |
| **Same pipeline API** | `@dsl.pipeline`, `command()`, `Input/Output` — identical |
| **Multi-model operations** | Two pipelines, two models, two endpoints — all managed |
| **Data-agnostic MLOps** | The platform doesn't care if it's text or tabular |

---

## Workshop Complete!

You've built **two complete ML systems** on Azure ML, demonstrating that
MLOps principles apply regardless of data type or ML problem.